## Merge TranscriptFormer ESM2 embeddings (mouse + human) and map gene names

1. Collect all gene names from raw datasets
2. Load TranscriptFormer ESM2 (2560-dim) embeddings for mouse + human
3. Bi-directionally map short names <-> Ensembl IDs via mygene
4. Intersect with required genes, report coverage per dataset, save to `data/other/esm/`

In [1]:
import os
import h5py
import torch
import numpy as np
import anndata as ad
import mygene
from pathlib import Path

/home/igor/miniconda3/envs/modeling/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


### Step 1: Collect all gene names from raw datasets

In [2]:
DATA_ROOT = Path("/home/igor/noise_scaling/data")
ESM_DATA = Path("/home/igor/noise_scaling/modeling/esm/tokenization_evaluation/data")
OUTPUT_PATH = Path("/home/igor/noise_scaling/data/other/esm/merged_esm_embeddings.pt")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

DATASET_NAMES = [d.name for d in DATA_ROOT.iterdir()
                 if d.is_dir() and (d / "raw" / "raw.h5ad").exists()]
DATASET_NAMES.sort()
print(f"Found {len(DATASET_NAMES)} datasets: {DATASET_NAMES}")

Found 4 datasets: ['PBMC', 'larry', 'merfish', 'shendure']


In [3]:
required_gene_names = set()
dataset_gene_names = {}  # per-dataset for coverage check later

for ds_name in DATASET_NAMES:
    path = DATA_ROOT / ds_name / "raw" / "raw.h5ad"
    adata = ad.read_h5ad(path, backed="r")

    genes = set(adata.var_names.tolist())
    # also include ensembl_id column if present and different
    if "ensembl_id" in adata.var.columns:
        genes |= set(adata.var["ensembl_id"].tolist())
    # also include gene_short_name if present
    if "gene_short_name" in adata.var.columns:
        genes |= set(adata.var["gene_short_name"].tolist())

    dataset_gene_names[ds_name] = genes
    required_gene_names |= genes
    print(f"  {ds_name}: {len(genes)} unique gene names")
    adata.file.close()

print(f"\nTotal unique gene names across all datasets: {len(required_gene_names)}")

  PBMC: 20729 unique gene names


  larry: 25289 unique gene names
  merfish: 483 unique gene names


  shendure: 91013 unique gene names

Total unique gene names across all datasets: 123949


### Step 2: Load TranscriptFormer ESM2 embeddings (2560-dim) for mouse + human

In [4]:
def load_h5_embeddings(path):
    emb = {}
    with h5py.File(path, "r") as f:
        gene_names = [k.decode("utf-8") if isinstance(k, bytes) else k for k in f["keys"][:]]
        for name in gene_names:
            emb[name] = torch.tensor(f["arrays"][name][:], dtype=torch.float32)
    return emb

tf_mouse = load_h5_embeddings(ESM_DATA / "all_embeddings" / "mus_musculus_gene.h5")
print(f"TranscriptFormer mouse (2560-dim): {len(tf_mouse)} genes")

tf_human = load_h5_embeddings(ESM_DATA / "all_embeddings" / "homo_sapiens_gene.h5")
print(f"TranscriptFormer human (2560-dim): {len(tf_human)} genes")

TranscriptFormer mouse (2560-dim): 22178 genes


TranscriptFormer human (2560-dim): 23823 genes


In [5]:
# Merge mouse + human into a single flat dict
flattened_merged_esm = {}

for name, tensor in tf_mouse.items():
    flattened_merged_esm[name] = tensor

for name, tensor in tf_human.items():
    if name not in flattened_merged_esm:
        flattened_merged_esm[name] = tensor

print(f"Merged ESM dict: {len(flattened_merged_esm)} unique gene keys")
print(f"  All embeddings are 2560-dim: {all(v.shape[0] == 2560 for v in flattened_merged_esm.values())}")

Merged ESM dict: 46001 unique gene keys
  All embeddings are 2560-dim: True


### Step 3: Build Ensembl ID <-> short name mappings via mygene

In [6]:
mg = mygene.MyGeneInfo()

all_keys = list(flattened_merged_esm.keys())
mouse_ensembl = [k for k in all_keys if k.startswith("ENSMUSG")]
human_ensembl = [k for k in all_keys if k.startswith("ENSG")]
short_names = [k for k in all_keys if not k.startswith("ENSMUSG") and not k.startswith("ENSG")]

print(f"Mouse Ensembl IDs to resolve: {len(mouse_ensembl)}")
print(f"Human Ensembl IDs to resolve: {len(human_ensembl)}")
print(f"Short names to resolve:       {len(short_names)}")

Mouse Ensembl IDs to resolve: 22178
Human Ensembl IDs to resolve: 23823
Short names to resolve:       0


In [7]:
# Batch query: Ensembl ID -> symbol
ensembl_to_short = {}
short_to_ensembl = {}

# Mouse ensembl -> symbol
if mouse_ensembl:
    results = mg.querymany(mouse_ensembl, scopes="ensembl.gene", fields="symbol",
                           species="mouse", returnall=True, verbose=False)
    for r in results["out"]:
        if "symbol" in r and not r.get("notfound"):
            ensembl_to_short[r["query"]] = r["symbol"]
    print(f"Mouse ensembl->symbol: {len([r for r in results['out'] if 'symbol' in r])}/{len(mouse_ensembl)} mapped")

# Human ensembl -> symbol
if human_ensembl:
    results = mg.querymany(human_ensembl, scopes="ensembl.gene", fields="symbol",
                           species="human", returnall=True, verbose=False)
    for r in results["out"]:
        if "symbol" in r and not r.get("notfound"):
            ensembl_to_short[r["query"]] = r["symbol"]
    print(f"Human ensembl->symbol: {len([r for r in results['out'] if 'symbol' in r])}/{len(human_ensembl)} mapped")

# Short name -> ensembl (try mouse first, then human; also try with alias scope for RIKEN names)
if short_names:
    # Try proper case first
    results = mg.querymany(short_names, scopes="symbol,alias", fields="ensembl.gene",
                           species="mouse", returnall=True, verbose=False)
    mapped_short = 0
    for r in results["out"]:
        if not r.get("notfound") and "ensembl" in r:
            eid = r["ensembl"]
            if isinstance(eid, list):
                eid = eid[0]
            if isinstance(eid, dict) and "gene" in eid:
                short_to_ensembl[r["query"]] = eid["gene"]
                mapped_short += 1
    print(f"Short->mouse ensembl: {mapped_short}/{len(short_names)} mapped")

    # For unmapped, also try case-corrected (title case for mouse)
    unmapped_short = [s for s in short_names if s not in short_to_ensembl]
    if unmapped_short:
        title_cased = [s.capitalize() if s.isupper() else s for s in unmapped_short]
        results = mg.querymany(title_cased, scopes="symbol,alias", fields="ensembl.gene",
                               species="mouse", returnall=True, verbose=False)
        extra = 0
        for r in results["out"]:
            if not r.get("notfound") and "ensembl" in r:
                eid = r["ensembl"]
                if isinstance(eid, list):
                    eid = eid[0]
                if isinstance(eid, dict) and "gene" in eid:
                    # store under original casing
                    orig = unmapped_short[title_cased.index(r["query"])] if r["query"] in title_cased else r["query"]
                    short_to_ensembl[orig] = eid["gene"]
                    extra += 1
        print(f"Short->mouse ensembl (title-cased): {extra} extra mapped")

print(f"\nFinal: ensembl_to_short={len(ensembl_to_short)}, short_to_ensembl={len(short_to_ensembl)}")

Mouse ensembl->symbol: 22147/22178 mapped


Human ensembl->symbol: 23122/23823 mapped

Final: ensembl_to_short=45247, short_to_ensembl=0


### Step 4: "Humanize" — duplicate embeddings under both short name and Ensembl ID

In [8]:
def is_ensembl_id(name):
    return name.startswith("ENSMUSG") or name.startswith("ENSG")

flattened_merged_esm_humanized = dict(flattened_merged_esm)  # start with a copy
added = 0

for original_name, embedding in list(flattened_merged_esm.items()):
    if is_ensembl_id(original_name):
        # ensembl -> find short name (and uppercase variant for larry-style datasets)
        short = ensembl_to_short.get(original_name)
        if short:
            for variant in [short, short.upper()]:
                if variant not in flattened_merged_esm_humanized:
                    flattened_merged_esm_humanized[variant] = embedding
                    added += 1
    else:
        # short name -> find ensembl id
        eid = short_to_ensembl.get(original_name)
        if eid and eid not in flattened_merged_esm_humanized:
            flattened_merged_esm_humanized[eid] = embedding
            added += 1

print(f"Original keys: {len(flattened_merged_esm)}")
print(f"Added {added} alias keys")
print(f"Humanized dict: {len(flattened_merged_esm_humanized)} total keys")

Original keys: 46001
Added 47618 alias keys
Humanized dict: 93619 total keys


### Step 5: Intersect with required gene names, check coverage, save

In [9]:
available_keys = set(flattened_merged_esm_humanized.keys())
intersection = required_gene_names & available_keys

final_esm = {k: flattened_merged_esm_humanized[k] for k in intersection}

print(f"Required gene names:  {len(required_gene_names)}")
print(f"Available ESM keys:   {len(available_keys)}")
print(f"Intersection (final): {len(final_esm)}")
print(f"Missing:              {len(required_gene_names - available_keys)}")

Required gene names:  123949
Available ESM keys:   93619
Intersection (final): 61260
Missing:              62689


In [10]:
# Per-dataset coverage report
print(f"{'Dataset':<12} {'Total genes':>12} {'Mapped':>8} {'Missing':>8} {'Coverage':>10}")
print("-" * 54)

for ds_name in DATASET_NAMES:
    ds_genes = dataset_gene_names[ds_name]
    mapped = ds_genes & available_keys
    missing = ds_genes - available_keys
    coverage = len(mapped) / len(ds_genes) * 100 if ds_genes else 0
    print(f"{ds_name:<12} {len(ds_genes):>12} {len(mapped):>8} {len(missing):>8} {coverage:>9.1f}%")

    # Show a few missing genes for debugging
    if missing and len(missing) <= 10:
        print(f"  missing: {sorted(missing)}")
    elif missing:
        print(f"  missing (first 10): {sorted(missing)[:10]}")

Dataset       Total genes   Mapped  Missing   Coverage
------------------------------------------------------
PBMC                20729    14899     5830      71.9%
  missing (first 10): ['A1BG-AS1', 'A2M-AS1', 'AAED1', 'AARS', 'AATBC', 'ABALON', 'ABCF2.1', 'ABHD15-AS1', 'AC000032.1', 'AC000068.1']
larry               25289    18146     7143      71.8%
  missing (first 10): ['0610006L08RIK', '0610007P14RIK', '0610009B22RIK', '0610009E02RIK', '0610009L18RIK', '0610009O20RIK', '0610010F05RIK', '0610010K14RIK', '0610011F06RIK', '0610012D04RIK']
merfish               483      479        4      99.2%
  missing: ['FPR-S1', 'GPR1', 'GRAMD3', 'P2YR13']
shendure            91013    40856    50157      44.9%
  missing (first 10): ['00R_AC107638.2', '00R_Pgap2', '0610005C13Rik', '0610006L08Rik', '0610007P14Rik', '0610009B22Rik', '0610009E02Rik', '0610009L18Rik', '0610009O20Rik', '0610010F05Rik']


In [11]:
# Save final dict
torch.save(final_esm, OUTPUT_PATH)
print(f"Saved {len(final_esm)} gene embeddings to {OUTPUT_PATH}")

# Quick sanity check
loaded = torch.load(OUTPUT_PATH, map_location="cpu")
print(f"Reload check: {len(loaded)} keys")
print(f"Embedding dims: {set(v.shape[0] for v in list(loaded.values())[:100])}")

Saved 61260 gene embeddings to /home/igor/noise_scaling/data/other/esm/merged_esm_embeddings.pt


Reload check: 61260 keys
Embedding dims: {2560}
